<a href="https://colab.research.google.com/github/AshrfCode/Anan-Tirgulem/blob/main/HW2_CAT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **SETUP & INSTALLATIONS**

In [ ]:
!pip -q install --no-cache-dir "pillow<12" gradio_client google-generativeai sentence-transformers faiss-cpu nltk bs4 firebase-admin

import os
import re
import json
import base64
import io
import tempfile
import requests
from collections import defaultdict
import numpy as np
from bs4 import BeautifulSoup
from PIL import Image

# ML & NLP
from gradio_client import Client, handle_file
import nltk
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sentence_transformers import SentenceTransformer
import faiss
import google.generativeai as genai

# Colab & Firebase
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import userdata, output
import IPython
from IPython.display import display, HTML

# Download NLTK resources
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# --- GEMINI SETUP ---
try:
    GEMINI_API_KEY =
    genai.configure(api_key=GEMINI_API_KEY)
    llm_model = genai.GenerativeModel('gemini-2.5-flash')
    print("✅ Gemini 2.5 Flash connected.")
except Exception:
    print("⚠️ Please add 'GEMINI_API_KEY' to your Colab Secrets for the RAG feature.")
    llm_model = None

# --- FIREBASE SETUP ---
db = None
if not firebase_admin._apps:
    try:
        key_content =
        if key_content:
            key_dict = json.loads(key_content)
            cred = credentials.Certificate(key_dict)
            firebase_admin.initialize_app(cred)
            db = firestore.client()
            print("✅ Firestore Database ready")
    except Exception as e:
        print(f"⚠️ Firestore connection failed! The exact error is: {e}")

# **SEARCH ENGINE**

In [ ]:
import json
import re
from collections import defaultdict
import requests
from bs4 import BeautifulSoup
from nltk.stem import PorterStemmer

# Assuming you have firebase_admin imported in your environment:
# import firebase_admin
# from firebase_admin import credentials, firestore
# from google.colab import userdata

class AcademicSearchEngine:
    def __init__(self, firebase_cred_path="serviceAccountKey.json", stop_words=None):
        """Initialize the search engine with custom stop words, a stemmer, and Firebase"""
        self.urls = [
            "https://link.springer.com/content/pdf/10.1007/1-4020-2607-2_12?pdf=chapter%20toc",
            "https://link.springer.com/content/pdf/10.1007/s10341-025-01743-7.pdf",
            "https://link.springer.com/protocol/10.1007/978-1-0716-4686-1_14",
            "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0160470",
            "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0322189"
        ]

        self.pages = []
        self.inverted_index = defaultdict(list)  # stemmed_word -> [(doc_id, frequency), ...]

        # NLP Setup: Purely Custom Stop Words
        if stop_words is None:
            # Your custom default list if nothing is passed
            self.stop_words = {
              "the","a","an","and","or","but",
              "to","of","in","on","at","for","from","by","with","as",
              "is","are","was","were","be","been","being",
              "this","that","these","those",
              "it","its","they","them","their","we","our","you","your",
              "i","me","my","he","him","his","she","her",
              "not","no","do","does","did","doing"
            }
        else:
            self.stop_words = set(stop_words)

        self.stemmer = PorterStemmer()

        # Firebase Setup (Using Colab Secrets)
        try:
            if not firebase_admin._apps:
                firebase_secret = userdata.get('FIREBASE')
                cred_dict = json.loads(firebase_secret)
                cred = credentials.Certificate(cred_dict)
                firebase_admin.initialize_app(cred)

            self.db = firestore.client()
            self.firebase_connected = True
            print("Firebase initialized successfully via Colab Secrets.")
        except Exception as e:
            self.firebase_connected = False

    def fetch_articles(self):
        """Fetch content from the predefined list of academic URLs"""
        headers = {
            'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36'
        }

        for idx, url in enumerate(self.urls):
            doc_id = str(idx + 1)
            try:
                response = requests.get(url, headers=headers, timeout=15)
                if response.status_code == 200:
                    soup = BeautifulSoup(response.text, 'html.parser')
                    paragraphs = soup.find_all('p')
                    text = ' '.join([p.get_text(strip=True) for p in paragraphs])

                    self.pages.append({
                        'id': doc_id,
                        'url': url,
                        'content': text
                    })
            except Exception as e:
                print(f"Error fetching Document {doc_id}: {str(e)}")

    def preprocess_text(self, text):
        """Normalize, remove custom stop words, and apply stemming"""
        words = re.findall(r'[a-zA-Z]+', text.lower())
        processed_words = []

        for word in words:
            if word not in self.stop_words and len(word) > 2:
                stemmed_word = self.stemmer.stem(word)
                processed_words.append(stemmed_word)

        return processed_words

    def build_index(self):
        """Build the inverted index mapping stemmed terms to document IDs"""
        self.inverted_index.clear()
        for page in self.pages:
            processed_words = self.preprocess_text(page['content'])
            term_counts = defaultdict(int)
            for word in processed_words:
                term_counts[word] += 1
            for term, count in term_counts.items():
                self.inverted_index[term].append({'doc_id': page['id'], 'freq': count})

    def upload_to_firestore(self):
        """Upload the inverted index to Firebase Firestore with strict tracking"""
        if not self.firebase_connected:
            print("❌ Upload Aborted: Firebase is not connected! Check your initialization credentials.")
            return

        print(f"🚀 Starting batch upload of {len(self.inverted_index)} terms...")

        batch = self.db.batch()
        collection_ref = self.db.collection('inverted_index')
        count = 0

        try:
            for term, postings in self.inverted_index.items():
                doc_ref = collection_ref.document(term)
                doc_links = []
                for posting in postings:
                    page = next((p for p in self.pages if p['id'] == posting['doc_id']), None)
                    if page:
                        formatted_link = f"Doc {page['id']}: {page['url']}"
                        if formatted_link not in doc_links:
                            doc_links.append(formatted_link)

                data_to_upload = {'term': term, 'DocIDs': doc_links}
                batch.set(doc_ref, data_to_upload)
                count += 1

                if count % 400 == 0:
                    batch.commit()
                    print(f"📦 Successfully committed batch: {count} documents uploaded so far...")
                    batch = self.db.batch()

            if count % 400 != 0:
                batch.commit()

            print(f"🎉 Success! Total documents written to Firestore: {count}")

        except Exception as e:
            print(f"❌ Firestore Write Error occurred: {str(e)}")

    def get_context(self, content, original_query_words):
        """Finds a snippet of text surrounding the matched words"""
        content_lower = content.lower()
        for word in original_query_words:
            match = re.search(r'(.{0,40}\b' + re.escape(word) + r'\b.{0,40})', content_lower)
            if match:
                return "..." + match.group(1).replace('\n', ' ').strip() + "..."
        return "Context not found."

    def search(self, query, num_results=5):
        """Search pages using the stemmed inverted index"""
        query_stems = self.preprocess_text(query)
        original_query_words = [w for w in re.findall(r'[a-zA-Z]+', query.lower()) if w not in self.stop_words]

        if not query_stems: return []

        page_scores = defaultdict(lambda: {'matches': 0, 'total_freq': 0})
        for stem in query_stems:
            postings = self.inverted_index.get(stem, [])
            for posting in postings:
                doc_id = posting['doc_id']
                freq = posting['freq']
                page_scores[doc_id]['matches'] += 1
                page_scores[doc_id]['total_freq'] += freq

        ranked_results = [
            (doc_id, scores['matches'], scores['total_freq'])
            for doc_id, scores in page_scores.items()
        ]
        ranked_results.sort(key=lambda x: (x[1], x[2]), reverse=True)

        results = []
        for doc_id, matches, total_freq in ranked_results[:num_results]:
            page = next(p for p in self.pages if p['id'] == doc_id)
            context = self.get_context(page['content'], original_query_words)
            results.append({
                'doc_id': doc_id, 'url': page['url'],
                'matching_stems': matches, 'total_frequency': total_freq,
                'context': context
            })
        return results

# ==========================================
# INITIALIZATION CODE
# ==========================================

print("🔍 Initializing Custom Search Engine...")

# Leaving the parentheses empty automatically uses the fallback list
# of default custom words we set up inside the class!
engine = AcademicSearchEngine()
engine.fetch_articles()

if engine.pages:
    engine.build_index()
    print("✅ Engine initialized and index built successfully!")
    print(f"Total unique words in index: {len(engine.inverted_index)}")

    #We documnted these lines so it doesnt upload always to the database when we run the cell
    #print("⏳ Uploading to database...")
    #engine.upload_to_firestore()
    #print("✅ Upload complete!")

# **RAG**

In [ ]:
CUSTOM_STOPWORDS = {
    "the","a","an","and","or","but","to","of","in","on","at","for","from","by",
    "with","as","is","are","was","were","be","been","being","this","that","these",
    "those","it","its","they","them","their","we","our","you","your","i","me","my",
    "not","no","do","does","did","plant","disease"
}

stemmer = PorterStemmer()

def preprocess_text_rag(text: str):
    text = text.lower()
    tokens = word_tokenize(text)
    return [stemmer.stem(tok) for tok in tokens if tok.isalpha() and tok not in CUSTOM_STOPWORDS]

class InvertedIndexStore:
    def __init__(self):
        self.term_to_docids = defaultdict(set)
    def add_occurrence(self, term: str, doc_id: str):
        self.term_to_docids[term].add(doc_id)
    def to_required_format(self):
        return [{"term": t, "DocIDs": sorted(list(docids))} for t, docids in self.term_to_docids.items()]

class FirebasePlantRAG:
    def __init__(self, firestore_db):
        self.db = firestore_db
        self.embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
        self.faiss_index = None
        self.docs_meta = {}
        self.vector_doc_ids = []

    def upload_and_index_papers(self, papers_list):
        local_texts = []
        inv_index = InvertedIndexStore()

        for paper in papers_list:
            doc_id = paper["id"]
            doc_url = paper["url"]
            self.docs_meta[doc_id] = paper
            self.vector_doc_ids.append(doc_id)
            local_texts.append(paper["text"])

            terms = preprocess_text_rag(paper["title"] + " " + paper["text"])
            for t in set(terms):
                formatted_link = f"Doc {doc_id}: {doc_url}"
                inv_index.add_occurrence(t, formatted_link)

        embeddings = self.embedding_model.encode(local_texts, convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        self.faiss_index = faiss.IndexFlatIP(embeddings.shape[1])
        self.faiss_index.add(embeddings)

    def query_gemini(self, question, top_k=2):
        q_emb = self.embedding_model.encode([question], convert_to_numpy=True, normalize_embeddings=True).astype("float32")
        distances, indices = self.faiss_index.search(q_emb, top_k)

        context_chunks = []
        for idx in indices[0]:
            doc_id = self.vector_doc_ids[idx]
            meta = self.docs_meta[doc_id]
            context_chunks.append(f"Source: {meta['title']} ({meta['url']})\nContent: {meta['text']}")

        full_context = "\n\n".join(context_chunks)
        prompt = f"""You are an agricultural AI assistant specializing in strawberries. Answer the user's question using ONLY the context provided below. Cite your sources.

        Context:
        {full_context}

        Question: {question}"""

        try:
            response = llm_model.generate_content(prompt)
            return response.text, context_chunks
        except Exception as e:
            return f"Gemini API Error: {str(e)}", context_chunks

academic_papers = [
    {
        "id": "1",
        "url": "https://link.springer.com/content/pdf/10.1007/1-4020-2607-2_12?pdf=chapter%20toc",
        "title": "Strawberry Disease Management",
        "authors": "Louws et al.",
        "text": "This chapter provides a comprehensive overview of disease management strategies for strawberry crops, addressing both soilborne and foliar pathogens to enhance crop yield and health."
    },
    {
        "id": "2",
        "url": "https://link.springer.com/content/pdf/10.1007/s10341-025-01743-7.pdf",
        "title": "Environmental, Economic, Greenhouse Gas Emission and Energy Balance Analysis of Open-Field Strawberry Production under Highland Conditions in Türkiye",
        "authors": "Yılmaz et al.",
        "text": "The aim of this study was to define the economy, energy balance, and greenhouse gas emissions of strawberry growing in highland conditions, providing insights into the sustainability of open-field production."
    },
    {
        "id": "3",
        "url": "https://link.springer.com/protocol/10.1007/978-1-0716-4686-1_14",
        "title": "Protocol for Strawberry Research",
        "authors": "Springer Protocols",
        "text": "A detailed methodology and protocol for working with Fragaria (strawberry) species, outlining step-by-step laboratory and analytical procedures for researchers."
    },
    {
        "id": "4",
        "url": "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0160470",
        "title": "Metagenomic Analysis of Fungal Diversity on Strawberry Plants",
        "authors": "Abdelfattah et al.",
        "text": "A comprehensive metagenomic sequencing of strawberry plants revealed a highly diverse fungal microbiome. The study highlights how environmental factors influence the balance between harmless fungi and aggressive pathogens on strawberry leaves."
    },
    {
        "id": "5",
        "url": "https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0322189",
        "title": "Vision Transformers for precision crop monitoring and strawberry disease detection",
        "authors": "Chen et al.",
        "text": "We implemented a deep learning model based on Vision Transformers to automatically detect and classify leaf diseases in strawberries. The classification model achieved superior accuracy compared to standard CNNs, providing early detection of infections in smart agricultural systems."
    }
]

print("🤖 Initializing RAG System...")
rag = FirebasePlantRAG(firestore_db=db)
rag.upload_and_index_papers(academic_papers)

# BACKEND (IoT Data & HuggingFace Analysis)

In [ ]:
BASE_URL = "https://server-cloud-v645.onrender.com"

def strawberrfy_get_iot_data(feed="humidity", limit=10):
    allowed_feeds = {"humidity", "soil", "temperature"}
    if feed not in allowed_feeds: feed = "humidity"
    try: limit = max(1, min(int(limit), 100))
    except Exception: limit = 10

    try:
        response = requests.get(f"{BASE_URL}/history", params={"feed": feed, "limit": limit}, timeout=20)
        response.raise_for_status()
        return response.json()
    except Exception as e:
        return {"error": str(e), "feed": feed, "count": 0, "data": []}

HF_STRAWBERRY_MODEL_ID = "Azhar1924/strawberry-leaf-disease-classification"
strawberry_gradio_client = None
WARNING_TEXT = "PERINGATAN: Gambar tidak dikenali sebagai penyakit stroberi yang valid."

def get_strawberry_client():
    global strawberry_gradio_client
    if strawberry_gradio_client is None:
        strawberry_gradio_client = Client(HF_STRAWBERRY_MODEL_ID)
    return strawberry_gradio_client

def strawberry_advice_from_label(label):
    label_lower = str(label).lower()
    if "healthy" in label_lower: return {"diagnosis": "Healthy Strawberry Plant", "color": "var(--green)", "advice": "The model did not detect a strawberry leaf disease. Continue normal monitoring and care."}
    if "angular" in label_lower: return {"diagnosis": "Angular Leaf Spot", "color": "var(--amber)", "advice": "Possible angular leaf spot. Avoid wetting leaves, improve air circulation."}
    if "powdery" in label_lower: return {"diagnosis": "Powdery Mildew", "color": "var(--red)", "advice": "Possible powdery mildew. Improve ventilation, reduce humidity around the leaves."}
    if "blight" in label_lower: return {"diagnosis": "Leaf Blight", "color": "var(--amber)", "advice": "Possible leaf blight. Remove affected leaves and monitor if the disease spreads."}
    if "spot" in label_lower: return {"diagnosis": "Leaf Spot", "color": "var(--amber)", "advice": "Possible leaf spot disease. Keep leaves dry, improve airflow."}
    return {"diagnosis": str(label), "color": "var(--amber)", "advice": "The model detected a strawberry leaf condition. Check the plant and monitor it carefully."}

def convert_model_result_to_ui(result):
    disease_labels_zero = [
        {"raw_label": "Healthy", "label": "Healthy Strawberry Plant", "score": 100.0},
        {"raw_label": "Angular Leaf Spot", "label": "Angular Leaf Spot", "score": 0.0},
        {"raw_label": "Powdery Mildew", "label": "Powdery Mildew", "score": 0.0},
        {"raw_label": "Leaf Spot", "label": "Leaf Spot", "score": 0.0},
        {"raw_label": "Leaf Blight", "label": "Leaf Blight", "score": 0.0}
    ]

    if isinstance(result, str) and WARNING_TEXT in result:
        return {"error": None, "diagnosis": "Healthy Strawberry Plant", "color": "var(--green)", "advice": "The model did not recognize a valid disease.", "confidence": 100.0, "data": disease_labels_zero}

    if isinstance(result, dict):
        top_label = str(result.get("label", "Healthy"))
        confidences = result.get("confidences", [])

        if WARNING_TEXT in top_label or "PERINGATAN" in top_label or "tidak dikenali" in top_label or top_label.lower() == "unknown":
            return {"error": None, "diagnosis": "Healthy Strawberry Plant", "color": "var(--green)", "advice": "The model returned an unknown result.", "confidence": 100.0, "data": disease_labels_zero}

        diagnosis_info = strawberry_advice_from_label(top_label)
        cleaned_predictions = [{"raw_label": item.get("label", "Unknown"), "label": item.get("label", "Unknown"), "score": round(float(item.get("confidence", 0)) * 100, 2)} for item in confidences]

        if not cleaned_predictions:
            cleaned_predictions = [{"raw_label": top_label, "label": top_label, "score": 100.0}]

        return {"error": None, "diagnosis": diagnosis_info["diagnosis"], "color": diagnosis_info["color"], "advice": diagnosis_info["advice"], "confidence": cleaned_predictions[0]["score"], "model_label": top_label, "data": cleaned_predictions}

    return {"error": None, "diagnosis": "Healthy Strawberry Plant", "color": "var(--green)", "advice": "The model returned an unrecognized result.", "confidence": 100.0, "data": disease_labels_zero}

def strawberrfy_analyze_plant_image(image_data_url):
    try:
        if not image_data_url: return {"error": "No image was received.", "data": []}
        image_base64 = image_data_url.split(",", 1)[1] if "," in image_data_url else image_data_url
        image_bytes = base64.b64decode(image_base64)

        with tempfile.NamedTemporaryFile(delete=False, suffix=".png") as temp_img:
            temp_img.write(image_bytes)
            temp_img_path = temp_img.name

        try:
            client = get_strawberry_client()
            result = client.predict(handle_file(temp_img_path), api_name="/predict")
            return convert_model_result_to_ui(result)
        finally:
            if os.path.exists(temp_img_path): os.remove(temp_img_path)
    except Exception as e:
        return {"error": str(e), "data": []}

# **REGISTER COLAB BRIDGES**

In [ ]:
def py_basic_search(query):
    try: return IPython.display.JSON({"results": engine.search(query)})
    except Exception as e: return IPython.display.JSON({"error": str(e)})

def py_gemini_search(query):
    try:
        answer, sources = rag.query_gemini(query)
        return IPython.display.JSON({"answer": answer, "sources": sources})
    except Exception as e:
        return IPython.display.JSON({"answer": f"Error: {str(e)}", "sources": []})

output.register_callback('basic_search', py_basic_search)
output.register_callback('gemini_search', py_gemini_search)
output.register_callback('strawberrfy.get_iot_data', strawberrfy_get_iot_data)
output.register_callback('strawberrfy.analyze_plant_image', strawberrfy_analyze_plant_image)

print("✅ All Colab Bridges Active! Launching Integrated UI...")

# **INTEGRATED HTML/JS UI**

In [ ]:
from IPython.display import display, HTML

html_app = """
<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8"/>
<meta name="viewport" content="width=device-width, initial-scale=1.0"/>
<title>Strawberrfy - Complete Integration</title>
<link href="https://fonts.googleapis.com/css2?family=Nunito:wght@400;500;600;700&family=Plus+Jakarta+Sans:wght@600;700;800&family=Syne:wght@700;800;900&display=swap" rel="stylesheet"/>
<style>
  *{margin:0;padding:0;box-sizing:border-box}
  :root{ --bg:#f6fbf7; --surface:#ffffff; --surface2:#edf8f0; --surface3:#f9fcfa; --border:#d9eadf; --text:#1f2d25; --muted:#6f8176; --green:#1f9d55; --green-dark:#137a3d; --green-dim:#e7f8ee; --amber:#d88712; --amber-dim:#fff4dd; --red:#e14d57; --red-dim:#fde9eb; --blue:#2f80ed; --blue-dim:#e8f1ff; --shadow:0 10px 24px rgba(31,45,37,.08); --shadow-soft:0 4px 14px rgba(31,45,37,.06); }
  body{ font-family:'Nunito',sans-serif; background:radial-gradient(circle at top left, rgba(31,157,85,.14), transparent 34%), linear-gradient(180deg,#fbfffc 0%,var(--bg) 100%); color:var(--text); min-height:100vh; display:flex; flex-direction:column; }

  /* NAV */
  nav{ background:rgba(255,255,255,.92); border-bottom:1px solid var(--border); box-shadow:var(--shadow-soft); backdrop-filter:blur(10px); padding:0 24px; display:flex; align-items:center; gap:0; height:58px; position:sticky; top:0; z-index:100; overflow-x:auto; white-space:nowrap; }
  .nav-brand{ font-family:'Plus Jakarta Sans',sans-serif; font-size:20px; font-weight:800; color:var(--green-dark); margin-right:32px; letter-spacing:-.2px; cursor:pointer; user-select:none; transition:transform .15s,color .15s; }
  .nav-brand:hover{transform:scale(1.04);color:var(--green)}
  .nav-tabs{display:flex;gap:4px;flex:1}
  .nav-tab{ background:transparent; border:1px solid transparent; color:var(--muted); font-family:'Nunito',sans-serif; font-size:12px; font-weight:700; padding:8px 14px; border-radius:999px; cursor:pointer; transition:all .15s; display:flex; align-items:center; gap:6px; }
  .nav-tab:hover{color:var(--green-dark);background:var(--green-dim);border-color:var(--border)}
  .nav-tab.active{color:var(--green-dark);background:var(--green-dim);border-color:#bfe6cc}
  .nav-dot{width:6px;height:6px;border-radius:50%;background:var(--green);animation:blink 2s infinite}
  @keyframes blink{0%,100%{opacity:1}50%{opacity:.3}}

  /* STRAWBERRY RAIN */
  .strawberry-rain-layer{position:fixed;inset:0;pointer-events:none;z-index:9999;overflow:hidden}
  .falling-strawberry{position:absolute;top:-50px;animation-name:strawberryFall;animation-timing-function:linear;animation-fill-mode:forwards;filter:drop-shadow(0 3px 4px rgba(0,0,0,.22))}
  @keyframes strawberryFall{ 0%{transform:translateY(-70px) rotate(0deg);opacity:0} 10%{opacity:1} 100%{transform:translateY(calc(100vh + 90px)) rotate(420deg);opacity:.95} }

  /* MAIN & SCREENS */
  main{flex:1;padding:30px 24px;max-width:960px;width:100%;margin:0 auto}
  .screen{display:none;animation:fadeIn .2s ease} .screen.active{display:block}
  @keyframes fadeIn{from{opacity:0;transform:translateY(6px)}to{opacity:1;transform:translateY(0)}}

  .page-title{font-family:'Plus Jakarta Sans',sans-serif;font-size:22px;font-weight:800;color:var(--text);margin-bottom:5px;letter-spacing:-.35px}
  .page-sub{font-size:12px;color:var(--muted);margin-bottom:24px;letter-spacing:.2px}
  .card{background:var(--surface);border:1px solid var(--border);border-radius:16px;padding:20px;box-shadow:var(--shadow-soft)}
  .label{font-size:10px;color:var(--muted);text-transform:uppercase;letter-spacing:1.4px;margin-bottom:7px;font-weight:800}
  .badge{display:inline-flex;align-items:center;gap:5px;padding:4px 10px;border-radius:20px;font-size:11px;font-weight:800}
  .badge-green{background:var(--green-dim);color:var(--green-dark);border:1px solid #bfe6cc}
  .badge-amber{background:var(--amber-dim);color:#9a5d00;border:1px solid #f2d39a}
  .badge-red{background:var(--red-dim);color:#b82f3a;border:1px solid #f4bbc1}
  .badge-blue{background:var(--blue-dim);color:#1d62b7;border:1px solid #bfd7ff}
  .section-title{font-size:11px;color:var(--muted);text-transform:uppercase;letter-spacing:1px;margin-bottom:12px;margin-top:20px;font-weight:800}

  /* ── DASHBOARD (Targel2) ── */
  .dashboard-top-row{display:flex;justify-content:space-between;align-items:flex-start;gap:16px;margin-bottom:18px;}
  .dashboard-game-btn{background:var(--green);border:none;color:white;font-family:'Plus Jakarta Sans',sans-serif;font-size:12px;font-weight:800;padding:8px 14px;border-radius:999px;cursor:pointer;box-shadow:var(--shadow-soft);transition:all .15s;}
  .dashboard-game-btn:hover{background:var(--green-dark);transform:translateY(-1px);}
  .overall-bar{background:var(--surface);border:1px solid var(--border);border-radius:16px;padding:15px 18px;display:flex;align-items:center;gap:14px;margin-bottom:20px;border-left:5px solid var(--green);box-shadow:var(--shadow-soft);}
  .overall-bar.warn{border-left-color:var(--amber)} .overall-bar.danger{border-left-color:var(--red)}
  .status-pulse{width:10px;height:10px;border-radius:50%;background:var(--green);box-shadow:0 0 0 4px var(--green-dim);flex-shrink:0}
  .status-pulse.warn{background:var(--amber);box-shadow:0 0 0 4px var(--amber-dim)} .status-pulse.danger{background:var(--red);box-shadow:0 0 0 4px var(--red-dim)}
  .sensor-grid{display:grid;grid-template-columns:repeat(3, 1fr);gap:14px;margin-bottom:20px}
  .sensor-card{background:linear-gradient(180deg,var(--surface) 0%,var(--surface3) 100%);border:1px solid var(--border);border-radius:16px;padding:17px 20px;position:relative;overflow:hidden;box-shadow:var(--shadow-soft);}
  .sensor-card::after{content:'';position:absolute;top:0;left:0;right:0;height:4px;background:var(--accent-c)}
  .sensor-val{font-family:'Plus Jakarta Sans',sans-serif;font-size:26px;font-weight:700;color:var(--accent-c);line-height:1.1;}
  .sensor-unit{font-size:12px;color:var(--muted);margin-left:3px;font-weight:700}
  .alerts-card{background:var(--surface);border:1px solid var(--border);border-radius:16px;padding:16px 18px;box-shadow:var(--shadow-soft)}
  .alert-row{font-size:12px;color:var(--muted);padding:8px 0;border-bottom:1px solid var(--border);display:flex;align-items:flex-start;gap:8px;line-height:1.6}
  .alert-row:last-child{border-bottom:none}
  .history-bar{display:flex;gap:4px;align-items:flex-end;height:55px;margin-top:4px}
  .hbar{flex:1;background:var(--green-dim);border-radius:4px;min-height:4px;transition:all .3s}

  /* ── UPLOAD (Targel2) ── */
  .upload-zone{border:1.5px dashed #afdcc0;border-radius:18px;padding:42px 24px;text-align:center;background:rgba(255,255,255,.78);cursor:pointer;transition:all .2s;margin-bottom:16px;box-shadow:var(--shadow-soft)}
  .upload-zone:hover{border-color:var(--green);background:var(--green-dim);transform:translateY(-1px)}
  .upload-icon{font-size:36px;margin-bottom:10px;opacity:.85}
  .upload-text{font-size:13px;color:var(--muted);margin-bottom:8px;font-weight:700}
  .upload-btn{background:var(--green);border:1px solid var(--green);color:#fff;font-family:'Nunito',sans-serif;font-size:12px;font-weight:800;padding:9px 22px;border-radius:999px;cursor:pointer;margin-top:10px;}
  .disease-grid{display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-bottom:20px}
  .disease-tag{background:var(--surface);border:1px solid var(--border);border-radius:12px;padding:10px 13px;font-size:11px;color:var(--muted);display:flex;align-items:center;gap:6px;box-shadow:var(--shadow-soft)}
  .result-box{display:none;background:var(--surface);border:1px solid var(--border);border-radius:16px;padding:20px;margin-top:16px;box-shadow:var(--shadow-soft)}
  .result-box.show{display:block;animation:fadeIn .3s ease}
  .result-image{width:100%;max-width:220px;height:180px;object-fit:cover;border-radius:12px;border:2px solid var(--result-color, var(--green));margin-bottom:14px;display:block}
  .conf-item{margin-bottom:9px}
  .conf-label{display:flex;justify-content:space-between;font-size:11px;margin-bottom:4px;color:var(--muted);font-weight:700}
  .conf-bar-bg{background:#e9f2ec;border-radius:999px;height:6px}
  .conf-bar-fill{height:6px;border-radius:999px;transition:width .6s ease}
  .file-input{display:none}

  /* ── IOT SENSORS ── */
  .iot-status-row{display:flex;gap:10px;flex-wrap:wrap;margin-bottom:20px}
  .sensor-pill{background:var(--surface);border:1px solid var(--border);border-radius:20px;padding:7px 14px;font-size:11px;font-weight:800;color:var(--muted);display:flex;align-items:center;gap:6px;box-shadow:var(--shadow-soft)}
  .live-dot{width:7px;height:7px;border-radius:50%;background:var(--green);animation:blink 1.5s infinite;flex-shrink:0}
  .data-table{width:100%;border-collapse:collapse;font-size:12px;background:var(--surface)}
  .data-table th{font-size:10px;color:var(--muted);text-transform:uppercase;letter-spacing:.8px;padding:11px 12px;text-align:left;border-bottom:1px solid var(--border);background:var(--surface2)}
  .data-table td{padding:11px 12px;border-bottom:1px solid var(--border);color:var(--text)}
  .iot-control-card{background:var(--surface);border:1px solid var(--border);border-radius:16px;padding:18px;display:grid;grid-template-columns:1fr 1fr auto;gap:14px;align-items:end;margin-bottom:16px;box-shadow:var(--shadow-soft);}
  .iot-field label{display:block;font-size:11px;color:var(--muted);text-transform:uppercase;margin-bottom:7px;font-weight:800;}
  .iot-select{width:100%;background:var(--surface2);border:1px solid var(--border);border-radius:12px;padding:10px 12px;color:var(--text);font-family:'Plus Jakarta Sans',sans-serif;font-size:13px;outline:none;}
  .iot-range{width:100%;accent-color:var(--green);}
  .iot-fetch-btn{background:var(--green);border:1px solid var(--green);color:#ffffff;font-family:'Plus Jakarta Sans',sans-serif;font-size:13px;font-weight:800;padding:11px 20px;border-radius:999px;cursor:pointer;box-shadow:var(--shadow-soft);}
  .iot-status-msg{background:var(--surface);border:1px solid var(--border);border-left:5px solid var(--green);border-radius:14px;padding:12px 14px;font-size:12px;color:var(--muted);margin-bottom:14px;box-shadow:var(--shadow-soft);}
  .iot-value-badge{display:inline-flex;padding:5px 10px;border-radius:999px;background:var(--green-dim);color:var(--green-dark);font-weight:800;font-family:'Plus Jakarta Sans',sans-serif;}

  /* ── GAME (Targel2) ── */
  .game-card{background:linear-gradient(135deg,#ffffff 0%,#f0fff5 100%);border:1px solid #bfe6cc;border-radius:20px;padding:20px;margin-top:22px;box-shadow:var(--shadow);position:relative;overflow:hidden;}
  .game-card::before{content:'🍓';position:absolute;right:18px;top:12px;font-size:52px;opacity:.12;transform:rotate(-15deg);}
  .game-head{display:flex;justify-content:space-between;align-items:flex-start;gap:14px;margin-bottom:14px;}
  .game-title{font-family:'Plus Jakarta Sans',sans-serif;font-size:18px;font-weight:800;color:var(--green-dark);margin-bottom:4px;}
  .game-score{background:var(--green-dim);border:1px solid #bfe6cc;border-radius:999px;padding:7px 13px;font-family:'Plus Jakarta Sans',sans-serif;font-size:12px;font-weight:800;color:var(--green-dark);}
  .game-scenario{background:var(--surface);border:1px solid var(--border);border-radius:16px;padding:15px;margin:14px 0;font-size:13px;color:var(--text);line-height:1.7;}
  .game-options{display:grid;grid-template-columns:1fr 1fr;gap:10px;margin-top:12px;}
  .game-option{background:#ffffff;border:1px solid var(--border);border-radius:14px;padding:11px 12px;cursor:pointer;text-align:left;font-family:'Nunito',sans-serif;font-size:12px;font-weight:800;color:var(--text);transition:all .15s;box-shadow:var(--shadow-soft);}
  .game-option:hover:not(:disabled){border-color:var(--green);background:var(--green-dim);transform:translateY(-1px);}
  .game-option:disabled{cursor:not-allowed;opacity:.75;}
  .game-option.correct{background:var(--green-dim);border-color:var(--green);color:var(--green-dark);}
  .game-option.wrong{background:var(--red-dim);border-color:#f4bbc1;color:#b82f3a;}
  .game-feedback{margin-top:13px;padding:12px 14px;border-radius:14px;font-size:12px;line-height:1.7;background:var(--surface);border:1px solid var(--border);color:var(--muted);}
  .game-feedback.good{background:var(--green-dim);border-color:#bfe6cc;color:var(--green-dark);}
  .game-feedback.bad{background:var(--red-dim);border-color:#f4bbc1;color:#b82f3a;}
  .game-actions{display:flex;gap:10px;flex-wrap:wrap;margin-top:14px;}
  .game-btn{background:var(--green);border:1px solid var(--green);color:#fff;font-family:'Plus Jakarta Sans',sans-serif;font-size:12px;font-weight:800;padding:9px 15px;border-radius:999px;cursor:pointer;transition:all .15s;box-shadow:var(--shadow-soft);}
  .game-btn:hover{background:var(--green-dark);transform:translateY(-1px);}
  .game-btn.secondary{background:var(--surface);color:var(--green-dark);border-color:#bfe6cc;}
  .game-progress{height:8px;background:#e8f1ec;border-radius:999px;overflow:hidden;margin-top:13px;}
  .game-progress-fill{height:100%;width:0%;background:linear-gradient(90deg,var(--green),#7bdc9a);transition:width .3s ease;}

  /* ── SEARCH & RAG (Untitled4) ── */
  .search-row{display:flex;gap:10px;margin-bottom:20px}
  .search-input{flex:1;background:var(--surface);border:1px solid var(--border);border-radius:12px;padding:11px 14px;color:var(--text);font-family:'Nunito',sans-serif;font-size:13px;outline:none;transition:border .15s,box-shadow .15s;box-shadow:var(--shadow-soft)}
  .search-input:focus{border-color:var(--green);box-shadow:0 0 0 4px var(--green-dim)}
  .search-btn{background:var(--green);border:1px solid var(--green);color:#fff;font-family:'Nunito',sans-serif;font-size:12px;font-weight:800;padding:10px 20px;border-radius:999px;cursor:pointer;transition:all .15s;white-space:nowrap;box-shadow:var(--shadow-soft)}
  .search-btn:hover{background:var(--green-dark);color:#fff}
  .result-card{background:var(--surface);border:1px solid var(--border);border-left:4px solid var(--green);border-radius:0 14px 14px 0;padding:15px 16px;margin-bottom:10px;animation:fadeIn .2s ease;box-shadow:var(--shadow-soft)}
  .result-term{font-family:'Plus Jakarta Sans',sans-serif;font-size:14px;font-weight:800;color:var(--green-dark);margin-bottom:6px}
  .result-docs{font-size:11px;color:var(--muted);margin-bottom:6px}
  .result-papers{font-size:13px;color:var(--text);line-height:1.6}
  .no-result{color:var(--muted);font-size:13px;text-align:center;padding:30px}
  .search-result-count{font-size:11px;color:var(--muted);margin-bottom:14px;font-weight:700}

  footer{border-top:1px solid var(--border);background:rgba(255,255,255,.72);padding:13px 24px;text-align:center;font-size:10px;color:var(--muted)}
</style>
</head>
<body>

<div id="strawberry-rain-layer" class="strawberry-rain-layer"></div>

<nav>
  <div class="nav-brand" onclick="showScreen('dashboard', document.querySelectorAll('.nav-tab')[0]); rainStrawberries(45)" title="Go to Dashboard">🍓 Strawberrfy</div>
  <div class="nav-tabs">
    <button class="nav-tab active" onclick="showScreen('dashboard',this)"><span class="nav-dot"></span> Dashboard</button>
    <button class="nav-tab" onclick="showScreen('upload',this)">📷 Plant Analysis</button>
    <button class="nav-tab" onclick="showScreen('iot',this)">📡 IoT Sensors</button>
    <button class="nav-tab" onclick="showScreen('search',this)">🔍 Search Engine</button>
    <button class="nav-tab" onclick="showScreen('rag',this)">🤖 AI Chat (RAG)</button>
  </div>
</nav>

<main>

  <div id="screen-dashboard" class="screen active">
    <div class="dashboard-top-row">
      <div>
        <div class="page-title">Plant Health Dashboard</div>
        <div class="page-sub">Real-time overview of your strawberry field — live IoT sensor data</div>
      </div>
      <button class="dashboard-game-btn" onclick="goToFarmGame()">🎮 Play Game</button>
    </div>
    <div class="overall-bar" id="overall-bar">
      <div class="status-pulse" id="overall-pulse"></div>
      <div>
        <div class="label">Overall Status</div>
        <div style="font-family:'Plus Jakarta Sans',sans-serif;font-size:15px;font-weight:800" id="overall-text">HEALTHY</div>
      </div>
      <div style="margin-left:auto;font-size:11px;color:var(--muted)" id="dash-time"></div>
    </div>

    <div class="sensor-grid" id="sensor-grid"></div>

    <div class="alerts-card" id="alerts-card">
      <div class="label">Active Alerts</div>
      <div id="alerts-list"></div>
    </div>

    <div class="section-title">Sensor History (Last 8 Readings)</div>

    <div class="label" style="font-size:9px">Soil Moisture (%)</div>
    <div class="card" style="padding:10px 18px; margin-bottom:12px">
      <div class="history-bar" id="history-bar-soil"></div>
      <div style="display:flex;justify-content:space-between;font-size:10px;color:var(--muted);margin-top:6px">
        <span>History Start</span><span>Now</span>
      </div>
    </div>

    <div class="label" style="font-size:9px">Temperature (°C)</div>
    <div class="card" style="padding:10px 18px; margin-bottom:12px">
      <div class="history-bar" id="history-bar-temp"></div>
      <div style="display:flex;justify-content:space-between;font-size:10px;color:var(--muted);margin-top:6px">
        <span>History Start</span><span>Now</span>
      </div>
    </div>

    <div class="label" style="font-size:9px">Air Humidity (%)</div>
    <div class="card" style="padding:10px 18px">
      <div class="history-bar" id="history-bar-hum"></div>
      <div style="display:flex;justify-content:space-between;font-size:10px;color:var(--muted);margin-top:6px">
        <span>History Start</span><span>Now</span>
      </div>
    </div>
  </div>

  <div id="screen-upload" class="screen">
    <div class="page-title">AI Plant Image Analysis</div>
    <div class="page-sub">Upload a strawberry leaf photo to detect diseases and get treatment advice</div>
    <div class="upload-zone" onclick="document.getElementById('file-input').click()">
      <div class="upload-icon">📷</div>
      <div class="upload-text">Click to upload a leaf image</div>
      <div style="font-size:11px;color:var(--muted)">JPG · PNG · WEBP</div>
      <button class="upload-btn" onclick="event.stopPropagation();document.getElementById('file-input').click()">Choose File</button>
    </div>
    <input type="file" id="file-input" class="file-input" accept="image/*" onchange="analyzeImage(event)"/>
    <div class="section-title">Detectable Conditions</div>
    <div class="disease-grid">
      <div class="disease-tag"><span style="color:var(--green)">●</span> Healthy Plant</div>
      <div class="disease-tag"><span style="color:var(--red)">●</span> Powdery Mildew</div>
      <div class="disease-tag"><span style="color:var(--amber)">●</span> Leaf Scorch</div>
      <div class="disease-tag"><span style="color:var(--red)">●</span> Gray Mold (Botrytis)</div>
      <div class="disease-tag"><span style="color:var(--amber)">●</span> Angular Leaf Spot</div>
    </div>
    <div class="result-box" id="result-box">
      <div class="label" style="margin-bottom:12px">Analysis Result</div>
      <img id="result-img" class="result-image" src="" alt="uploaded leaf"/>
      <div id="result-diagnosis" style="margin-bottom:14px"></div>
      <div class="section-title" style="margin-top:0">Model Confidence Scores</div>
      <div id="conf-bars"></div>
      <div id="result-advice" style="background:var(--surface2);border-radius:8px;padding:12px 14px;font-size:12px;color:var(--muted);line-height:1.7;margin-top:14px"></div>
    </div>
  </div>

  <div id="screen-iot" class="screen">
    <div class="page-title">IoT Sensor Data Sampling</div>
    <div class="page-sub">Live readings from the Strawberrfy cloud server</div>
    <div class="iot-status-row">
      <div class="sensor-pill"><div class="live-dot"></div> Cloud Server</div>
      <div class="sensor-pill">🌡 Temperature</div>
      <div class="sensor-pill">💧 Air Humidity</div>
      <div class="sensor-pill">🌱 Soil Moisture</div>
    </div>
    <div class="iot-control-card">
      <div class="iot-field">
        <label for="iot-feed">Choose Sensor Feed</label>
        <select id="iot-feed" class="iot-select">
          <option value="humidity">Air Humidity</option>
          <option value="soil">Soil Moisture</option>
          <option value="temperature">Temperature</option>
        </select>
      </div>
      <div class="iot-field">
        <label for="iot-limit">Samples: <span id="iot-limit-value">10</span></label>
        <input type="range" id="iot-limit" class="iot-range" value="10" min="1" max="100" step="1" oninput="document.getElementById('iot-limit-value').textContent=this.value"/>
      </div>
      <button class="iot-fetch-btn" onclick="fetchIoTData()">Get Live Data</button>
    </div>
    <div id="iot-message" class="iot-status-msg">Choose a sensor feed and click “Get Live Data”.</div>
    <div class="card" style="padding:0;overflow:hidden">
      <table class="data-table" id="iot-table">
        <thead>
          <tr><th>#</th><th>Time</th><th>Feed</th><th>Value</th><th>Unit</th><th>Status</th></tr>
        </thead>
        <tbody id="iot-body"></tbody>
      </table>
    </div>
    <div style="display:flex;justify-content:space-between;margin-top:12px;font-size:10px;color:var(--muted)">
      <span id="iot-session"></span><span id="iot-count"></span>
    </div>

    <div id="farm-game-card" class="game-card">
      <div class="game-head">
        <div>
          <div class="game-title">🎮 Strawberry Farm Challenge</div>
          <div class="game-sub">Train the farm manager to choose the best action based on sensor situations.</div>
        </div>
        <div class="game-score" id="game-score">Score: 0 / 0</div>
      </div>
      <div class="game-scenario" id="game-scenario">Click “New Challenge” to start.</div>
      <div class="game-options" id="game-options"></div>
      <div class="game-feedback" id="game-feedback">Choose an answer to get instant feedback.</div>
      <div class="game-progress"><div class="game-progress-fill" id="game-progress-fill"></div></div>
      <div class="game-actions">
        <button class="game-btn" onclick="newFarmChallenge(false)">🎲 New Challenge</button>
        <button class="game-btn secondary" onclick="newFarmChallenge(true)">📡 Use Live Sensor Values</button>
        <button class="game-btn secondary" onclick="resetFarmGame()">↻ Reset Score</button>
      </div>
    </div>
  </div>

  <div id="screen-search" class="screen">
    <div class="page-title">Keyword Search Engine</div>
    <div class="page-sub">Query the academic inverted index for keyword matches and stems.</div>
    <div class="search-row">
      <input class="search-input" id="basic-search-input" placeholder="e.g. soil moisture content" onkeydown="if(event.key==='Enter')doBasicSearch()"/>
      <button class="search-btn" onclick="doBasicSearch()">Search Index →</button>
    </div>
    <div id="basic-search-results-wrap"></div>
  </div>

  <div id="screen-rag" class="screen">
    <div class="page-title">Research AI Assistant</div>
    <div class="page-sub">Ask questions and get AI-synthesized answers from academic papers via Gemini.</div>
    <div class="search-row">
      <input class="search-input" id="rag-search-input" placeholder="e.g. Which model is best for precision crop monitoring?" onkeydown="if(event.key==='Enter')doRagSearch()"/>
      <button class="search-btn" onclick="doRagSearch()">Ask Gemini →</button>
    </div>
    <div id="rag-search-results-wrap"></div>
  </div>

</main>

<footer>Strawberrfy v1.0 · Braude Academic College · Strawberry Plant Monitoring System</footer>

<script>
// ─── NAVIGATION ───
function showScreen(id, btn) {
  document.querySelectorAll('.screen').forEach(s => s.classList.remove('active'))
  document.querySelectorAll('.nav-tab').forEach(t => t.classList.remove('active'))
  document.getElementById('screen-' + id).classList.add('active')
  btn.classList.add('active')
}

function goToFarmGame() {
  const tabs = document.querySelectorAll('.nav-tab');
  showScreen('iot', tabs[2]);
  setTimeout(() => {
    const gameCard = document.getElementById('farm-game-card');
    if (gameCard) gameCard.scrollIntoView({ behavior: 'smooth', block: 'start' });
  }, 200);
}

// ─── UTILITIES ───
function rand(min, max) { return Math.round(Math.random() * (max - min) + min) }
function randf(min, max, dec) { return parseFloat((Math.random() * (max - min) + min).toFixed(dec)) }
function now() { return new Date().toLocaleTimeString('en-GB', {hour:'2-digit',minute:'2-digit'}) }
function status(val, low, high) { if (val < low) return 'low'; if (val > high) return 'high'; return 'ok' }
function statusColor(s) { return s === 'ok' ? 'var(--green)' : s === 'low' ? 'var(--amber)' : 'var(--red)' }
function statusBadge(s) {
  if (s === 'ok') return '<span class="badge badge-green">● OK</span>'
  if (s === 'low') return '<span class="badge badge-amber">▼ LOW</span>'
  return '<span class="badge badge-red">▲ HIGH</span>'
}
function escapeHtml(text) {
  return String(text).replaceAll("&", "&amp;").replaceAll("<", "&lt;").replaceAll(">", "&gt;").replaceAll('"', "&quot;").replaceAll("'", "&#039;");
}

function rainStrawberries(count = 45) {
  const layer = document.getElementById('strawberry-rain-layer')
  if (!layer) return
  for (let i = 0; i < count; i++) {
    const berry = document.createElement('div')
    berry.className = 'falling-strawberry'
    berry.textContent = '🍓'
    berry.style.left = (Math.random() * 100) + 'vw'
    berry.style.fontSize = rand(18, 34) + 'px'
    berry.style.animationDelay = (Math.random() * 1.2) + 's'
    berry.style.animationDuration = randf(2.4, 5.2, 1) + 's'
    layer.appendChild(berry)
    berry.addEventListener('animationend', () => berry.remove())
  }
}

// ─── TARGEL2 EXACT COLAB JSON PARSING ───
function parseMaybeJson(value) {
  let current = value;
  for (let i = 0; i < 5; i++) {
    if (typeof current !== "string") return current;
    let text = current.trim();
    try { current = JSON.parse(text); continue; } catch (e) {}
    try {
      const fixed = text.replace(/\\bNone\\b/g, "null").replace(/\\bTrue\\b/g, "true").replace(/\\bFalse\\b/g, "false").replace(/'/g, '"');
      current = JSON.parse(fixed); continue;
    } catch (e) {}
    return current;
  }
  return current;
}

function findObjectWithDataList(value, depth = 0) {
  if (depth > 8 || value === null || value === undefined) return null;
  const parsed = parseMaybeJson(value);
  if (Array.isArray(parsed)) return {feed: "unknown", count: parsed.length, data: parsed};
  if (typeof parsed !== "object") return null;
  if (Array.isArray(parsed.data)) return parsed;
  if (parsed.data && typeof parsed.data === "object") {
    if (parsed.data["application/json"] !== undefined) {
      const found = findObjectWithDataList(parsed.data["application/json"], depth + 1);
      if (found) return found;
    }
    if (parsed.data["text/plain"] !== undefined) {
      const found = findObjectWithDataList(parsed.data["text/plain"], depth + 1);
      if (found) return found;
    }
  }
  for (const key of ["payload", "result", "response", "body", "content", "value"]) {
    if (parsed[key] !== undefined) {
      const found = findObjectWithDataList(parsed[key], depth + 1);
      if (found) return found;
    }
  }
  for (const key of Object.keys(parsed)) {
    const found = findObjectWithDataList(parsed[key], depth + 1);
    if (found) return found;
  }
  return null;
}

function extractColabCallbackData(response) {
  return findObjectWithDataList(response) || parseMaybeJson(response);
}

// ─── DASHBOARD & IOT ───
let latestDashboardSamples = { temperature: null, humidity: null, soil: null };

function toSensorNumber(sample) {
  if (!sample || sample.value === undefined || sample.value === null) return null;
  const num = Number(sample.value); return Number.isNaN(num) ? null : num;
}
function sensorValueText(sample) {
  if (!sample || sample.value === undefined || sample.value === null) return '--';
  return escapeHtml(sample.value);
}
function sensorBadge(s) {
  if (s === 'missing') return '<span class="badge badge-blue">⏳ Loading</span>';
  return statusBadge(s);
}
function sensorAccentColor(s) {
  if (s === 'missing') return 'var(--blue)';
  return statusColor(s);
}

// UPDATED: buildDashboard now generates numbers on top of the bars
function buildDashboard(values = null, soilHistory = [], tempHistory = [], humHistory = []) {
  const hasValues = values !== null;
  const tempNum = toSensorNumber(hasValues ? values.temperature : null);
  const humNum = toSensorNumber(hasValues ? values.humidity : null);
  const soilNum = toSensorNumber(hasValues ? values.soil : null);

  const tS = tempNum === null ? 'missing' : status(tempNum, 18, 28);
  const hS = humNum === null ? 'missing' : status(humNum, 50, 80);
  const sS = soilNum === null ? 'missing' : status(soilNum, 40, 80);

  const realStatuses = [tS, hS, sS].filter(s => s !== 'missing');
  const hasRealData = realStatuses.length > 0;
  const allOk = hasRealData && realStatuses.every(s => s === 'ok');
  const anyDanger = realStatuses.some(s => s === 'high');

  const bar = document.getElementById('overall-bar');
  const pulse = document.getElementById('overall-pulse');
  const txt = document.getElementById('overall-text');

  if (!hasRealData) {
    bar.className = 'overall-bar warn'; pulse.className = 'status-pulse warn';
    txt.textContent = 'LOADING — Waiting for live IoT sensor data'; txt.style.color = 'var(--amber)';
  } else {
    bar.className = 'overall-bar' + (allOk ? '' : anyDanger ? ' danger' : ' warn');
    pulse.className = 'status-pulse' + (allOk ? '' : anyDanger ? ' danger' : ' warn');
    txt.textContent = allOk ? 'HEALTHY — Live IoT sensors normal' : anyDanger ? 'CRITICAL — Immediate action needed' : 'ATTENTION — Some live values out of range';
    txt.style.color = allOk ? 'var(--green)' : anyDanger ? 'var(--red)' : 'var(--amber)';
  }

  document.getElementById('dash-time').textContent = 'Updated ' + now();

  const sensors = [
    {label:'🌡 Temperature', val:sensorValueText(hasValues ? values.temperature : null), unit:'°C', s:tS},
    {label:'💧 Air Humidity', val:sensorValueText(hasValues ? values.humidity : null), unit:'%', s:hS},
    {label:'🌱 Soil Moisture', val:sensorValueText(hasValues ? values.soil : null), unit:'%', s:sS},
  ];

  document.getElementById('sensor-grid').innerHTML = sensors.map(s => `
    <div class="sensor-card" style="--accent-c:${sensorAccentColor(s.s)}">
      <div class="label">${s.label}</div>
      <div><span class="sensor-val">${s.val}</span><span class="sensor-unit">${s.unit}</span></div>
      <div style="margin-top:8px">${sensorBadge(s.s)}</div>
    </div>`).join('');

  const alerts = [];
  if (!hasRealData) alerts.push({icon:'📡', msg:'Loading live IoT readings...'});
  else {
    if (tS !== 'ok' && tS !== 'missing') alerts.push({icon:'🌡', msg:`Temperature is ${tS.toUpperCase()} (${tempNum}°C)`});
    if (hS !== 'ok' && hS !== 'missing') alerts.push({icon:'💧', msg:`Air humidity is ${hS.toUpperCase()} (${humNum}%)`});
    if (sS !== 'ok' && sS !== 'missing') alerts.push({icon:'🌱', msg:`Soil moisture is ${sS.toUpperCase()} (${soilNum}%)`});
  }

  document.getElementById('alerts-list').innerHTML = alerts.length ? alerts.map(a => `<div class="alert-row"><span class="alert-icon">${a.icon}</span><span>${a.msg}</span></div>`).join('') : '<div class="alert-row" style="color:var(--green)">✓ All live IoT sensors normal</div>';

  // Helper to render bars dynamically with permanent text values
  const renderHistory = (data, color, maxExpected) => {
    const rawValues = data.map(sample => Number(sample.value)).filter(v => !Number.isNaN(v)).reverse();
    return rawValues.length ? rawValues.map(v => {
      const h = Math.max(4, Math.min(40, (v / maxExpected) * 40));
      return `
        <div style="flex:1; display:flex; flex-direction:column; justify-content:flex-end; align-items:center; gap:4px;">
          <span style="font-size:9px; font-weight:800; color:${color}; line-height:1">${v}</span>
          <div style="width:100%; height:${h}px; background:${color}22; border-top:2px solid ${color}; border-radius:3px; transition:all .3s"></div>
        </div>`;
    }).join('') : '<div style="font-size:12px;color:var(--muted);padding:10px">Waiting for history...</div>';
  };

  document.getElementById('history-bar-soil').innerHTML = renderHistory(soilHistory, '#1f9d55', 100);
  document.getElementById('history-bar-temp').innerHTML = renderHistory(tempHistory, '#d88712', 50);
  document.getElementById('history-bar-hum').innerHTML  = renderHistory(humHistory, '#2f80ed', 100);
}

async function fetchIoTData() {
  const feed = document.getElementById("iot-feed").value;
  const limit = document.getElementById("iot-limit").value;
  const msg = document.getElementById("iot-message");
  msg.innerHTML = `Loading data...`;
  try {
    const res = await google.colab.kernel.invokeFunction("strawberrfy.get_iot_data", [feed, Number(limit)], {});
    const data = extractColabCallbackData(res);
    document.getElementById("iot-body").innerHTML = data.data.map((d,i) => `
      <tr>
        <td>${i+1}</td>
        <td>${d.created_at || d.time || '--'}</td>
        <td>${feed}</td>
        <td><span class="iot-value-badge">${escapeHtml(d.value)}</span></td>
        <td>${feed==='temperature'?'°C':'%'}</td>
        <td>${statusBadge(status(d.value, feed==='temperature'?18:40, feed==='temperature'?28:80))}</td>
      </tr>
    `).join('');
    msg.innerHTML = `Showing ${data.data.length} samples.`;
    document.getElementById("iot-count").textContent = `${data.data.length} readings`;
    refreshDashboardFromIoT();
  } catch(e) {
    msg.innerHTML = `Error: ${e.message}`;
  }
}

async function refreshDashboardFromIoT() {
  try {
    const tData = await google.colab.kernel.invokeFunction("strawberrfy.get_iot_data", ["temperature", 1], {});
    const hData = await google.colab.kernel.invokeFunction("strawberrfy.get_iot_data", ["humidity", 1], {});
    const sData = await google.colab.kernel.invokeFunction("strawberrfy.get_iot_data", ["soil", 1], {});

    const sHist = await google.colab.kernel.invokeFunction("strawberrfy.get_iot_data", ["soil", 8], {});
    const tHist = await google.colab.kernel.invokeFunction("strawberrfy.get_iot_data", ["temperature", 8], {});
    const hHist = await google.colab.kernel.invokeFunction("strawberrfy.get_iot_data", ["humidity", 8], {});

    latestDashboardSamples = {
      temperature: extractColabCallbackData(tData).data[0] || null,
      humidity: extractColabCallbackData(hData).data[0] || null,
      soil: extractColabCallbackData(sData).data[0] || null
    };

    buildDashboard(
      latestDashboardSamples,
      extractColabCallbackData(sHist).data || [],
      extractColabCallbackData(tHist).data || [],
      extractColabCallbackData(hHist).data || []
    );
  } catch (e) { console.log(e); }
}

// ─── TARGEL2 EXACT IMAGE UPLOAD LOGIC ───
async function analyzeImage(e) {
  const file = e.target.files[0];
  if (!file) return;

  const reader = new FileReader();

  reader.onload = async function(ev) {
    const src = ev.target.result;

    const resultBox = document.getElementById('result-box');
    const resultImg = document.getElementById('result-img');
    const diagnosisBox = document.getElementById('result-diagnosis');
    const confBars = document.getElementById('conf-bars');
    const adviceBox = document.getElementById('result-advice');

    resultImg.src = src;
    resultImg.style.borderColor = 'var(--blue)';

    diagnosisBox.innerHTML = `
      <div class="label">Analysis Result</div>
      <div style="font-family:'Plus Jakarta Sans',sans-serif;font-size:17px;font-weight:800;color:var(--blue)">
        🤖 Analyzing with Hugging Face model...
      </div>
    `;

    confBars.innerHTML = '';
    adviceBox.innerHTML = 'Please wait while the AI model analyzes the leaf image.';
    resultBox.classList.add('show');

    try {
      if (
        typeof google === "undefined" ||
        !google.colab ||
        !google.colab.kernel
      ) {
        throw new Error("Colab backend is not available. Run this full cell inside Google Colab.");
      }

      const callbackResponse = await google.colab.kernel.invokeFunction(
        "strawberrfy.analyze_plant_image",
        [src],
        {}
      );

      const data = extractColabCallbackData(callbackResponse);

      if (!data) {
        throw new Error("No analysis data was returned from the Colab callback.");
      }

      if (data.error) {
        throw new Error(data.error);
      }

      renderHuggingFacePlantResult(src, data);

    } catch (error) {
      diagnosisBox.innerHTML = `
        <div class="label">Analysis Error</div>
        <div style="font-family:'Plus Jakarta Sans',sans-serif;font-size:17px;font-weight:800;color:var(--red)">
          ⚠ Could not analyze image
        </div>
      `;

      adviceBox.innerHTML = `
        <strong style="color:var(--text)">Error:</strong><br>
        ${escapeHtml(error.message)}
      `;

      resultImg.style.borderColor = 'var(--red)';
    }
  };

  reader.readAsDataURL(file);
}

function renderHuggingFacePlantResult(src, data) {
  const color = data.color || 'var(--green)';
  const predictions = Array.isArray(data.data) ? data.data : [];

  document.getElementById('result-img').src = src;
  document.getElementById('result-img').style.borderColor = color;

  document.getElementById('result-diagnosis').innerHTML = `
    <div class="label">Detected Condition</div>
    <div style="font-family:'Plus Jakarta Sans',sans-serif;font-size:17px;font-weight:800;color:${color}">
      🤖 ${escapeHtml(data.diagnosis || 'Unknown Result')}
    </div>
    <div style="font-size:11px;color:var(--muted);margin-top:6px;line-height:1.6">
      Model label: ${escapeHtml(data.model_label || 'Unknown')}<br>
      Confidence: ${escapeHtml(data.confidence || 0)}%
    </div>
  `;

  document.getElementById('conf-bars').innerHTML = predictions.map(pred => {
    const score = Number(pred.score || 0);
    const safeScore = Math.max(0, Math.min(score, 100));

    return `
      <div class="conf-item">
        <div class="conf-label">
          <span>${escapeHtml(pred.label)}</span>
          <span style="color:${color}">${safeScore.toFixed(1)}%</span>
        </div>
        <div class="conf-bar-bg">
          <div class="conf-bar-fill" style="width:${safeScore}%;background:${color}"></div>
        </div>
      </div>
    `;
  }).join('');

  document.getElementById('result-advice').innerHTML = `
    <strong style="color:var(--text)">AI Recommendation:</strong><br>
    ${escapeHtml(data.advice || 'No advice available.')}<br><br>
    <span style="font-size:11px;color:var(--muted)">
      Model used: ${escapeHtml(data.model_id || 'Hugging Face image classifier')}
    </span>
  `;
}

// ─── GAME ───
let farmGameState = { score: 0, total: 0, current: null, answered: false };
const FARM_ACTIONS = [
  {id:'increase-irrigation', text:'💧 Increase irrigation'},
  {id:'reduce-irrigation', text:'🚫 Reduce irrigation'},
  {id:'improve-ventilation', text:'🌬 Improve ventilation'},
  {id:'improve-airflow', text:'🍃 Improve air circulation'},
  {id:'monitor', text:'👀 Keep monitoring'},
  {id:'check-disease', text:'🔎 Check leaves for disease'},
];
const FARM_CHALLENGES = [
  {title:'Dry soil alert', scenario:'🌱 Soil moisture is LOW while temperature is normal.', correct:'increase-irrigation', explanation:'Low moisture needs water.'},
  {title:'Wet soil alert', scenario:'🌱 Soil moisture is HIGH.', correct:'reduce-irrigation', explanation:'High moisture needs less water.'},
  {title:'Hot greenhouse', scenario:'🌡 Temperature is HIGH.', correct:'improve-ventilation', explanation:'High temps need ventilation.'},
  {title:'Humidity risk', scenario:'💧 Air humidity is HIGH.', correct:'improve-airflow', explanation:'High humidity breeds fungus.'},
  {title:'Healthy field', scenario:'✅ All normal.', correct:'monitor', explanation:'Keep monitoring.'}
];

function newFarmChallenge(useLive) {
  let challenge = FARM_CHALLENGES[Math.floor(Math.random() * FARM_CHALLENGES.length)];
  if(useLive && latestDashboardSamples.temperature) {
    const s = toSensorNumber(latestDashboardSamples.soil);
    if(s && s < 40) challenge = FARM_CHALLENGES[0];
    else if(s && s > 80) challenge = FARM_CHALLENGES[1];
  }
  farmGameState.current = challenge; farmGameState.answered = false;
  document.getElementById('game-scenario').innerHTML = `<strong>${challenge.title}</strong><br>${challenge.scenario}`;
  document.getElementById('game-feedback').className = 'game-feedback'; document.getElementById('game-feedback').innerHTML = 'Choose the best action.';

  let wrong = FARM_ACTIONS.filter(a => a.id !== challenge.correct).sort(()=>Math.random() - 0.5).slice(0,3);
  let opts = [FARM_ACTIONS.find(a => a.id === challenge.correct), ...wrong].sort(()=>Math.random() - 0.5);

  document.getElementById('game-options').innerHTML = opts.map(o => `
    <button class="game-option" onclick="answerFarmChallenge('${o.id}', this)">${o.text}</button>
  `).join('');
}

function answerFarmChallenge(actionId, btn) {
  if(farmGameState.answered) return;
  farmGameState.answered = true; farmGameState.total++;
  const correct = farmGameState.current.correct;
  if(actionId === correct) { farmGameState.score++; btn.classList.add('correct'); document.getElementById('game-feedback').className='game-feedback good'; document.getElementById('game-feedback').innerHTML = `✅ Correct! ${farmGameState.current.explanation}`; }
  else { btn.classList.add('wrong'); document.getElementById('game-feedback').className='game-feedback bad'; document.getElementById('game-feedback').innerHTML = `❌ Wrong. ${farmGameState.current.explanation}`; }

  document.getElementById('game-score').innerText = `Score: ${farmGameState.score} / ${farmGameState.total}`;
  document.getElementById('game-progress-fill').style.width = Math.round((farmGameState.score/farmGameState.total)*100) + '%';
  document.querySelectorAll('.game-option').forEach(b => b.disabled = true);
}

function resetFarmGame() { farmGameState={score:0,total:0,current:null,answered:false}; document.getElementById('game-score').innerText='Score: 0 / 0'; document.getElementById('game-progress-fill').style.width='0%'; newFarmChallenge(); }
function initFarmChallenge() { newFarmChallenge(false); }

// ─── SEARCH & RAG (DECOUPLED FROM IOT EXTRACTION) ───
async function doBasicSearch() {
  const q = document.getElementById('basic-search-input').value.trim();
  if (!q) return;
  const wrap = document.getElementById('basic-search-results-wrap');
  wrap.innerHTML = '<div style="padding: 20px; font-weight: bold; color: var(--green);">🔍 Querying Inverted Index... Please wait.</div>';
  try {
    if (typeof google === 'undefined' || !google.colab || !google.colab.kernel) throw new Error("Colab kernel not accessible.");

    const res = await google.colab.kernel.invokeFunction('basic_search', [q], {});
    const data = res.data['application/json'] || res.data;

    if (data.error) throw new Error(data.error);

    let html = `<div class="search-result-count">→ Found ${data.results.length} documents for: <span style="color:var(--green)">${q}</span></div>`;
    if (data.results.length === 0) html += `<div class="no-result">No matches found in the inverted index for "${q}".</div>`;

    data.results.forEach((r) => {
      html += `
        <div class="result-card">
          <div class="result-term">📄 Document ${r.doc_id}</div>
          <div class="result-docs">URL: <a href="${r.url}" target="_blank" style="color:var(--blue)">${r.url}</a></div>
          <div class="result-docs" style="font-weight:bold; color:var(--amber)">Matches: ${r.matching_stems} stems | ${r.total_frequency} hits</div>
          <div class="result-papers" style="margin-top: 6px;"><i>${r.context}</i></div>
        </div>`;
    });
    wrap.innerHTML = html;
  } catch(e) {
    console.error("Bridge Error:", e);
    wrap.innerHTML = `<div class="no-result" style="color: var(--red);"><strong>Error:</strong> ${e.message}</div>`;
  }
}

async function doRagSearch() {
  const q = document.getElementById('rag-search-input').value.trim();
  if (!q) return;
  const wrap = document.getElementById('rag-search-results-wrap');
  wrap.innerHTML = '<div style="padding: 20px; font-weight: bold; color: var(--green);">🤖 Querying Gemini AI... Please wait.</div>';
  try {
    if (typeof google === 'undefined' || !google.colab || !google.colab.kernel) throw new Error("Colab kernel not accessible.");

    const res = await google.colab.kernel.invokeFunction('gemini_search', [q], {});
    const data = res.data['application/json'] || res.data;

    if (data.error) throw new Error(data.error);

    let html = `<div class="search-result-count">→ Results for: <span style="color:var(--green)">${q}</span></div>`;
    html += `
      <div class="result-card" style="border-left-color: var(--amber);">
        <div class="result-term">🤖 Gemini AI Synthesis</div>
        <div class="result-papers" style="margin-top: 10px;">${data.answer.replace(/\\n/g, '<br>')}</div>
      </div>`;

    data.sources.forEach((src, idx) => {
      html += `
        <div class="result-card">
          <div class="result-term">📄 Retrieved Source ${idx + 1}</div>
          <div class="result-papers" style="margin-top: 6px;">${src.replace(/\\n/g, '<br>')}</div>
        </div>`;
    });
    wrap.innerHTML = html;
  } catch(e) {
    console.error("Bridge Error:", e);
    wrap.innerHTML = `<div class="no-result" style="color: var(--red);"><strong>Error:</strong> ${e.message}</div>`;
  }
}

// ─── INIT ───
function initApp() {
  buildDashboard();
  initFarmChallenge();
  setTimeout(refreshDashboardFromIoT, 800);
  setTimeout(fetchIoTData, 1200);
  setTimeout(() => rainStrawberries(65), 300);
}

initApp()
</script>
</body>
</html>
"""

display(HTML(html_app))